# Task 2 — Ensemble (DQN + h18 planner) — end-to-end walkthrough

This notebook reproduces the **ensemble submission** for Phase 2 step-by-step. The pipeline:

| Section | What it does |
|---|---|
| 0 | Setup, toggle flags, imports |
| 1 | (Optional) Train the Exp I DQN — frame_stack=4 + DDQN + LR/2 + save-best + Boltzmann + h18 warm-start on difficulty 2, 400 episodes |
| 2 | Cross-difficulty evaluation of the DQN alone (matches the Task 2 grading) |
| 3 | Load the ensemble `agent.py` (DQN + h18 planner routing) |
| 4 | Cross-difficulty evaluation of the ensemble |
| 5 | Comparison table: baselines vs Task 1 DQN vs Task 2 DQN-only vs h18 alone vs ensemble |
| 6 | Package the `.zip` and verify it runs from a clean extraction |

Flags at the top of section 0 control whether each phase runs from scratch or reuses cached artifacts.

## 0 · Setup

In [9]:
# Toggle flags — choose what to run vs reuse from disk
RUN_TRAINING        = False   # True = train Exp I from scratch (~45 min GPU). False = reuse the saved checkpoint.
EVAL_DQN_ALONE      = True    # Cross-diff on the DQN-only submission (task2_I_fs4_diff2_ddqn_400)
EVAL_ENSEMBLE       = True    # Cross-diff on the ensemble submission (task2_ensemble)
BUILD_ZIP           = True    # Re-create submission_task2_ensemble.zip
VERIFY_ZIP          = True    # Extract the zip to a temp dir and run a sanity episode

# Names
DQN_SUBMISSION_NAME      = 'task2_I_fs4_diff2_ddqn_400'
ENSEMBLE_SUBMISSION_NAME = 'task2_ensemble'
CHECKPOINT_NAME          = 'task2_I_fs4_diff2_ddqn_400.pt'

In [10]:
from __future__ import annotations
import importlib.util, json, shutil, statistics, sys, tempfile, time, zipfile
from pathlib import Path

# Locate repo root and put the package on sys.path
for _c in (Path.cwd(), *Path.cwd().parents):
    if (_c / 'Assignment2' / 'src' / 'spacerace_dqn').exists():
        ASSIGNMENT_DIR = _c / 'Assignment2'
        break
else:
    raise RuntimeError('Cannot find Assignment2/src/spacerace_dqn — run this notebook from the repo')

sys.path.insert(0, str(ASSIGNMENT_DIR / 'src'))
sys.path.insert(0, str(ASSIGNMENT_DIR / 'starting_kit'))
DOCS_DIR        = ASSIGNMENT_DIR / 'docs'
SUBMISSIONS_DIR = ASSIGNMENT_DIR / 'submissions'

from spacerace_dqn import (
    Config, DQNInferencePolicy, Evaluator,
    SubmissionPackager, plot_training,
)
from spacerace_dqn.config2  import Config2
from spacerace_dqn.trainer2 import DQNTrainer2
from spacerace_dqn.env      import make_env, reset_env

import numpy as np
import torch

print(f'Repo root: {ASSIGNMENT_DIR}')
print(f'torch    : {torch.__version__}  cuda: {torch.cuda.is_available()}'
      f'{"  device=" + torch.cuda.get_device_name(0) if torch.cuda.is_available() else ""}')

Repo root: c:\Projects\DeepLearning\Assignment2
torch    : 2.6.0+cu124  cuda: True  device=NVIDIA GeForce RTX 3060 Laptop GPU


## 1 · Train the Exp I DQN (frame_stack=4 + DDQN + LR/2 + save-best)

All Task 2 ingredients in one run:

- Replay buffer cap=10 000 (assignment §2.1) — uniform FIFO via `deque(maxlen=10000)`
- Target network with hard updates every 800 grad-steps (assignment §2.3)
- Double DQN to control overestimation
- Boltzmann exploration `T: 5.0 → 0.05` over 80 000 env steps
- `learning_rate = 5e-5` (half of vanilla) for stability on diff 2
- Frame stacking (`frame_stack=4`, state shape `(12, 54, 39)`) — motion cues for random-debris layouts
- Heuristic warm-start with **h18** for 10 episodes (~6000 transitions of expert data in the buffer before any gradient step)
- `save_best` — restores the highest-scoring checkpoint at the end of training, not the noisy last episode

Set `RUN_TRAINING = True` (in section 0) to re-train from scratch. By default we reuse the existing checkpoint.

In [11]:
cfg_i = Config2(
    exploration               = 'boltzmann',
    episodes                  = 400,
    eval_every                = 20,
    eval_episodes             = 10,
    baseline_episodes         = 10,
    frame_stack               = 4,
    difficulty                = 2,
    learning_rate             = 5e-5,
    use_double_dqn            = True,
    target_update_freq        = 800,
    target_update_mode        = 'hard',
    temperature_start         = 5.0,
    temperature_end           = 0.05,
    temperature_decay_steps   = 80_000,
    heuristic_warmup_episodes = 10,
    submission_name           = DQN_SUBMISSION_NAME,
    checkpoint_name           = CHECKPOINT_NAME,
)

checkpoint_path = SUBMISSIONS_DIR / DQN_SUBMISSION_NAME / CHECKPOINT_NAME

if RUN_TRAINING:
    print('Training Exp I from scratch — expect ~45 min on a single GPU')
    trainer = DQNTrainer2(cfg_i, verbose=True)
    agent_i = trainer.train()
    final_i = trainer.final_eval()
    print(f'\nFinal eval (best-restored): mean={final_i["mean_score"]:.2f}  '
          f'std={final_i["std_score"]:.2f}  max_q={final_i["mean_max_q"]:.3f}')

    packager = SubmissionPackager(cfg_i)
    pkg = packager.package(
        agent_i,
        state_shape   = trainer.state_shape,
        history       = trainer.history,
        eval_history  = trainer.eval_history,
        baselines     = [],
        final_eval    = final_i,
        train_minutes = trainer.train_minutes,
    )
    plot_training(trainer.history, trainer.eval_history,
                  packager.submission_dir / 'training_curves.png')
    print('Packaged:', pkg['files']['zip'])
else:
    assert checkpoint_path.exists(), (
        f'Checkpoint not found at {checkpoint_path}. Set RUN_TRAINING=True to train from scratch.'
    )
    print(f'Reusing cached checkpoint: {checkpoint_path}')
    ckpt = torch.load(checkpoint_path, map_location='cpu', weights_only=False)
    h    = ckpt.get('history', [])
    eh   = ckpt.get('eval_history', [])
    print(f'Training history: {len(h)} train episodes, {len(eh)} eval points')
    if eh:
        best = max(eh, key=lambda r: (r.get('mean_score', -1), r.get('mean_max_q', 0.0)))
        print(f'  best eval observed during training:  '
              f'ep={best["episode"]}  mean={best["mean_score"]:.2f}  '
              f'max_q={best.get("mean_max_q", float("nan")):.3f}')

Reusing cached checkpoint: c:\Projects\DeepLearning\Assignment2\submissions\task2_I_fs4_diff2_ddqn_400\task2_I_fs4_diff2_ddqn_400.pt
Training history: 400 train episodes, 21 eval points
  best eval observed during training:  ep=340  mean=23.00  max_q=4.820


## 2 · Cross-difficulty evaluation — DQN alone

This is the **Task 2 deliverable**: the DQN was trained on difficulty 2 with replay + target net + DDQN + frame stacking. Below we evaluate the trained checkpoint on **all four difficulties** using the Codabench-mirrored evaluator (seeds 2026..2035, `include_semantic_info=False`).

In [12]:
if EVAL_DQN_ALONE:
    agent_py = SUBMISSIONS_DIR / DQN_SUBMISSION_NAME / 'agent.py'
    policy   = DQNInferencePolicy(agent_py)
    evaluator = Evaluator(Config())

    print(f"{'Diff':>4}  {'Mean':>7}  {'Std':>6}  {'Min':>5}  {'Max':>5}  {'Collisions':>10}")
    dqn_results = {}
    for diff in (0, 1, 2, 3):
        r = evaluator.run(policy, difficulty=diff, n_episodes=10,
                          base_seed=2026, include_semantic_info=False)
        dqn_results[diff] = r
        print(f"{diff:>4}  {r['mean_score']:7.2f}  {r['std_score']:6.2f}  "
              f"{r['min_score']:5.0f}  {r['max_score']:5.0f}  {r['mean_collisions']:10.2f}")
else:
    print('Skipped (EVAL_DQN_ALONE=False)')

Diff     Mean     Std    Min    Max  Collisions
   0     7.00    0.00      7      7       17.00
   1    24.00    0.00     24     24        3.00
   2    23.10    2.12     19     26        4.00
   3    19.60    1.62     16     22        6.00


## 3 · Load the ensemble agent

The ensemble `agent.py` lives in `submissions/task2_ensemble/`. It is self-contained (no project imports) so it can be submitted directly to Codabench.

Routing logic, summary:

```text
select_action(obs):
    1. Q(s) = trained DQN (frame_stack=4)
    2. a_planner = h18 full-tree (k=4, max_speed=3) on the same RGB obs
    3. if argmax(Q) == a_planner            -> use it
       elif |Q_up - Q_down| > 2.0           -> trust the DQN (strong opinion)
       else                                  -> defer to the planner
```

In [13]:
ensemble_agent_py = SUBMISSIONS_DIR / ENSEMBLE_SUBMISSION_NAME / 'agent.py'
ensemble_ckpt     = SUBMISSIONS_DIR / ENSEMBLE_SUBMISSION_NAME / CHECKPOINT_NAME

# Ensure the checkpoint is mirrored into the ensemble dir (must sit next to agent.py)
src = SUBMISSIONS_DIR / DQN_SUBMISSION_NAME / CHECKPOINT_NAME
if ensemble_ckpt.exists() and ensemble_ckpt.stat().st_size != src.stat().st_size:
    print(f'Refreshing ensemble checkpoint from {src.name}')
    shutil.copy2(src, ensemble_ckpt)
elif not ensemble_ckpt.exists():
    print(f'Copying checkpoint into ensemble dir: {src.name}')
    shutil.copy2(src, ensemble_ckpt)

spec = importlib.util.spec_from_file_location('ensemble_agent', ensemble_agent_py)
ens_mod = importlib.util.module_from_spec(spec)
spec.loader.exec_module(ens_mod)
ensemble_agent = ens_mod.Agent()
print(f'Loaded ensemble.Agent() from {ensemble_agent_py.name}')
print(f'  DQN input channels = {ens_mod.INPUT_CHANNELS}  (= 3 * FRAME_STACK = {ens_mod.FRAME_STACK})')
print(f'  Planner k          = {ens_mod.PLANNER_K}  max_speed = {ens_mod.PLANNER_MAX_SPEED}')
print(f'  Routing threshold  = {ens_mod.DISAGREEMENT_THRESHOLD}')

Loaded ensemble.Agent() from agent.py
  DQN input channels = 12  (= 3 * FRAME_STACK = 4)
  Planner k          = 4  max_speed = 3
  Routing threshold  = 2.0


## 4 · Cross-difficulty evaluation — ensemble

The ensemble agent is not a `Policy` subclass (it takes only `obs`, no `info` or `action_space` args) so we run it through a small driver loop instead of `Evaluator`. Same protocol: 10 episodes per difficulty, seeds 2026..2035, `include_semantic_info=False`.

In [14]:
def evaluate_ensemble(agent, difficulty: int, n_episodes: int = 10, base_seed: int = 2026):
    cfg = Config()
    env = make_env(cfg, difficulty=difficulty, include_semantic_info=False)
    scores, collisions, returns = [], [], []
    for ep in range(n_episodes):
        obs, info = reset_env(env, seed=base_seed + ep)
        agent.planner.reset()
        agent.frames.clear()
        done, steps, ep_return = False, 0, 0.0
        while not done and steps < cfg.max_steps_per_episode:
            a = agent.select_action(obs)
            obs, r, term, trunc, info = env.step(int(a))
            ep_return += float(r)
            done = bool(term or trunc); steps += 1
        scores.append(float(info.get('score', ep_return)))
        collisions.append(float(info.get('collisions', float('nan'))))
        returns.append(ep_return)
    env.close()
    return {
        'difficulty':      difficulty,
        'episodes':        n_episodes,
        'mean_score':      float(np.mean(scores)),
        'std_score':       float(np.std(scores)),
        'min_score':       float(np.min(scores)),
        'max_score':       float(np.max(scores)),
        'mean_return':     float(np.mean(returns)),
        'mean_collisions': float(np.nanmean(collisions)),
        'scores':          [float(s) for s in scores],
    }

ensemble_results = {}
if EVAL_ENSEMBLE:
    print(f"{'Diff':>4}  {'Mean':>7}  {'Std':>6}  {'Min':>5}  {'Max':>5}  {'Collisions':>10}  {'Wall':>5}")
    for diff in (0, 1, 2, 3):
        t0 = time.time()
        r  = evaluate_ensemble(ensemble_agent, diff, n_episodes=10, base_seed=2026)
        r['wall_seconds'] = round(time.time() - t0, 1)
        ensemble_results[diff] = r
        print(f"{diff:>4}  {r['mean_score']:7.2f}  {r['std_score']:6.2f}  "
              f"{r['min_score']:5.0f}  {r['max_score']:5.0f}  "
              f"{r['mean_collisions']:10.2f}  {r['wall_seconds']:5.1f}s")

    out = DOCS_DIR / f'{ENSEMBLE_SUBMISSION_NAME}_difficulty_eval.json'
    out.write_text(json.dumps({'submission': ENSEMBLE_SUBMISSION_NAME, 'results': ensemble_results}, indent=2), encoding='utf-8')
    print(f'\nSaved: {out}')
else:
    print('Skipped (EVAL_ENSEMBLE=False)')

Diff     Mean     Std    Min    Max  Collisions   Wall
   0    29.00    0.00     29     29        0.00  132.5s
   1    30.00    0.00     30     30        0.00  136.5s
   2    29.60    0.66     29     31        0.10  148.8s
   3    28.60    0.66     27     29        0.10  147.5s

Saved: c:\Projects\DeepLearning\Assignment2\docs\task2_ensemble_difficulty_eval.json


## 5 · Comparison: baselines vs DQN-only vs ensemble

Pull baseline numbers from cached JSONs to avoid re-running the baselines (each takes ~1 min).

In [18]:
def _load_json(p):
    try: return json.loads(Path(p).read_text(encoding='utf-8'))
    except Exception: return None

rows = []

# Static baselines — file is a dict {"0": [policy_entry, ...], "1": [...], ...}
bmd = _load_json(DOCS_DIR / 'baselines_multi_difficulty.json')
if bmd:
    by_policy = {}
    for diff_key, entries in bmd.items():
        d = int(diff_key)
        for e in entries:
            by_policy.setdefault(e['policy'], {})[d] = e['mean_score']
    for name in ('random', 'always_up', 'rgb_heuristic'):
        d = by_policy.get(name, {})
        rows.append((name, d.get(0), d.get(1), d.get(2), d.get(3)))

# Task 1 basic DQN
t1 = _load_json(DOCS_DIR / 'task1_basic_dqn_difficulty_eval.json')
if t1 and 'results' in t1:
    by_diff = {r['difficulty']: r['mean_score'] for r in t1['results']}
    rows.append(('Task 1 basic DQN', by_diff.get(0), by_diff.get(1), by_diff.get(2), by_diff.get(3)))

# Heuristic h18 — per-diff JSONs
h18 = {d: _load_json(DOCS_DIR / 'heuristic_experiments' / f'h18_full_tree_s3_diff{d}.json') for d in (0,1,2,3)}
if all(h18.values()):
    rows.append(('h18_full_tree_s3 (planner)', *[h18[d]['mean_score'] for d in (0,1,2,3)]))

# Task 2 DQN alone (Exp I) — from this notebook
if EVAL_DQN_ALONE:
    rows.append(('Task 2 DQN (Exp I, this notebook)',
                 *[dqn_results[d]['mean_score'] for d in (0,1,2,3)]))

# Ensemble — from this notebook
if EVAL_ENSEMBLE:
    rows.append(('Ensemble (DQN + h18, this notebook)',
                 *[ensemble_results[d]['mean_score'] for d in (0,1,2,3)]))

print(f"{'Policy':<38}  {'Diff 0':>7}  {'Diff 1':>7}  {'Diff 2':>7}  {'Diff 3':>7}")
print('-' * 78)
for name, d0, d1, d2, d3 in rows:
    def fmt(v): return f'{v:7.2f}' if isinstance(v, (int, float)) else '   --  '
    print(f"{name:<38}  {fmt(d0)}  {fmt(d1)}  {fmt(d2)}  {fmt(d3)}")

Policy                                   Diff 0   Diff 1   Diff 2   Diff 3
------------------------------------------------------------------------------
random                                     0.00     0.40     0.10     0.20
always_up                                 17.00    17.00    10.80    10.10
rgb_heuristic                             17.00    21.00    21.10    18.50
Task 1 basic DQN                          24.00    21.00    13.80    11.90
h18_full_tree_s3 (planner)                29.00    30.00    29.60    28.60
Task 2 DQN (Exp I, this notebook)          7.00    24.00    23.10    19.60
Ensemble (DQN + h18, this notebook)       29.00    30.00    29.60    28.60


## 6 · Package the submission and verify

Builds `submission_task2_ensemble.zip` with exactly two files: `agent.py` and the DQN checkpoint. The ZIP is then extracted into a temporary directory and one episode is rolled out to confirm Codabench-style loading works end-to-end.

In [16]:
submission_dir = SUBMISSIONS_DIR / ENSEMBLE_SUBMISSION_NAME
zip_path       = submission_dir / f'submission_{ENSEMBLE_SUBMISSION_NAME}.zip'

if BUILD_ZIP:
    if zip_path.exists():
        zip_path.unlink()
    with zipfile.ZipFile(zip_path, 'w', compression=zipfile.ZIP_DEFLATED) as zf:
        zf.write(submission_dir / 'agent.py', arcname='agent.py')
        zf.write(submission_dir / CHECKPOINT_NAME, arcname=CHECKPOINT_NAME)
    print(f'Built {zip_path.name}  ({zip_path.stat().st_size:,} bytes)')
    with zipfile.ZipFile(zip_path) as zf:
        for info in zf.infolist():
            print(f'   {info.filename:<40} {info.file_size:>10,} bytes')
else:
    print('Skipped (BUILD_ZIP=False)')
    print(f'Existing zip: {zip_path}  ({zip_path.stat().st_size:,} bytes)')

Built submission_task2_ensemble.zip  (633,177 bytes)
   agent.py                                     10,174 bytes
   task2_I_fs4_diff2_ddqn_400.pt               719,106 bytes


In [17]:
if VERIFY_ZIP:
    tmp = Path(tempfile.mkdtemp(prefix='ensemble_verify_'))
    try:
        with zipfile.ZipFile(zip_path) as zf:
            zf.extractall(tmp)
        print(f'Extracted to: {tmp}')

        # Load the extracted agent.py as if Codabench was loading it
        spec = importlib.util.spec_from_file_location('isolated_ensemble', tmp / 'agent.py')
        iso  = importlib.util.module_from_spec(spec); spec.loader.exec_module(iso)
        iso_agent = iso.Agent()
        print('Isolated Agent constructed.')

        # Run a single diff-2 episode
        env = make_env(Config(), difficulty=2, include_semantic_info=False)
        obs, _ = reset_env(env, seed=2026)
        iso_agent.planner.reset(); iso_agent.frames.clear()
        done, steps, t0 = False, 0, time.time()
        while not done and steps < 600:
            a = iso_agent.select_action(obs)
            assert a in (0, 1), f'invalid action {a!r}'
            obs, _, term, trunc, info = env.step(int(a))
            done = bool(term or trunc); steps += 1
        env.close()
        print(f'Isolated diff-2 episode (seed 2026): score={info.get("score")}  '
              f'wall={time.time()-t0:.1f}s  (limit: 30s/ep)')
    finally:
        shutil.rmtree(tmp, ignore_errors=True)
else:
    print('Skipped (VERIFY_ZIP=False)')

Extracted to: C:\Users\PCMULT~1\AppData\Local\Temp\ensemble_verify_tz5yupef
Isolated Agent constructed.
Isolated diff-2 episode (seed 2026): score=29  wall=14.0s  (limit: 30s/ep)


## Done

Submit `submissions/task2_ensemble/submission_task2_ensemble.zip` to Codabench Phase 2.

Expected score, based on the local→Codabench mapping observed across 3 prior submissions (`+0` to `+1.5`):

| Phase / Difficulty | Local mean | Codabench expected |
|---|---:|---|
| Phase 1 / diff 0 | 29.00 | ~29 |
| Phase 1 / diff 1 | 30.00 | ~30 |
| Phase 2 / diff 2 | 29.60 | ~29-31 (confirmed: 29.4 already submitted) |
| Phase 3 / diff 3 | 28.60 | ~28-30 |